# Step 1: 번호판 검출 (License Plate Detection)

YOLOv8을 사용하여 이미지에서 번호판 영역을 검출합니다.

## 실행 전 확인
1. **런타임 → 런타임 유형 변경 → GPU (T4)** 선택
2. 순서대로 셀 실행

## 1. 패키지 설치

In [ ]:
!pip install ultralytics scikit-learn -q
print("Installation complete!")

## 2. Kaggle 데이터셋 다운로드

`andrewmvd/car-plate-detection` 데이터셋 사용 (433개 이미지 + XML 라벨 포함)

In [ ]:
# Kaggle API 키 업로드
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API configured!")

In [ ]:
# 데이터 다운로드 (라벨 포함된 데이터셋)
!kaggle datasets download -d andrewmvd/car-plate-detection
!unzip -q car-plate-detection.zip -d data/
!ls -la data/
print("Download complete!")

### 데이터셋 정보

| 항목 | 내용 |
|------|------|
| 데이터셋 | andrewmvd/car-plate-detection |
| 이미지 수 | 433개 |
| 라벨 형식 | Pascal VOC XML |
| 클래스 | licence (번호판) |

## 3. 데이터 구조 확인

이 데이터셋은 **Pascal VOC XML 형식**으로 제공됩니다:
- `images/` - 이미지 파일 (.png)
- `annotations/` - XML 라벨 파일 (.xml)

In [ ]:
import os

# 데이터 폴더 구조 확인
print("=== 데이터 폴더 구조 ===")
for root, dirs, files in os.walk('data/'):
    level = root.replace('data/', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # 2단계까지만 표시
        subindent = ' ' * 2 * (level + 1)
        for f in files[:3]:
            print(f"{subindent}{f}")
        if len(files) > 3:
            print(f"{subindent}... ({len(files)} files total)")

print("\n=== 파일 유형별 개수 ===")

# 이미지 파일
img_files = []
for root, dirs, files in os.walk('data/'):
    for file in files:
        if file.lower().endswith(('.jpg', '.png', '.jpeg')):
            img_files.append(os.path.join(root, file))
print(f"이미지 파일: {len(img_files)}개")

# YOLO 라벨 파일 (.txt)
txt_files = []
for root, dirs, files in os.walk('data/'):
    for file in files:
        if file.endswith('.txt') and file != 'classes.txt':
            txt_files.append(os.path.join(root, file))
print(f"라벨 파일 (.txt): {len(txt_files)}개")

# XML 파일 (참고용)
xml_files = []
for root, dirs, files in os.walk('data/'):
    for file in files:
        if file.endswith('.xml'):
            xml_files.append(os.path.join(root, file))
print(f"XML 파일: {len(xml_files)}개")

# 데이터 형식 판별
if len(txt_files) > 0:
    print("\n✅ YOLO 형식 데이터셋 (txt 라벨)")
    label_format = 'yolo'
elif len(xml_files) > 0:
    print("\n✅ VOC/XML 형식 데이터셋")
    label_format = 'xml'
else:
    print("\n⚠️ 라벨 파일을 찾을 수 없습니다. 데이터 구조를 확인해주세요.")
    label_format = None

In [ ]:
# 샘플 이미지 확인
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

if len(img_files) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, img_path in zip(axes.flat, img_files[:6]):
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.set_title(os.path.basename(img_path)[:15])
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("이미지를 찾을 수 없습니다.")

## 4. XML → YOLO 형식 변환 및 Train/Val 분리

Pascal VOC XML 형식을 YOLO 형식으로 변환합니다.

In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split

# 출력 폴더 생성
os.makedirs('dataset/images/train', exist_ok=True)
os.makedirs('dataset/images/val', exist_ok=True)
os.makedirs('dataset/labels/train', exist_ok=True)
os.makedirs('dataset/labels/val', exist_ok=True)

# XML → YOLO 변환 함수
def convert_xml_to_yolo(xml_path):
    """XML 라벨을 YOLO 형식으로 변환"""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # 이미지 크기
    size = root.find('size')
    img_width = int(size.find('width').text)
    img_height = int(size.find('height').text)

    yolo_lines = []
    for obj in root.findall('object'):
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)

        # YOLO 형식 (정규화된 중심점 + 크기)
        center_x = ((xmin + xmax) / 2) / img_width
        center_y = ((ymin + ymax) / 2) / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height

        # class_id center_x center_y width height
        yolo_lines.append(f"0 {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}")

    return '\n'.join(yolo_lines)

# XML 파일 찾기
xml_files = []
for root_dir, dirs, files in os.walk('data/'):
    for file in files:
        if file.endswith('.xml'):
            xml_files.append(os.path.join(root_dir, file))

print(f"XML 파일: {len(xml_files)}개")

if len(xml_files) == 0:
    raise ValueError("XML 파일을 찾을 수 없습니다!")

# Train/Val 분리 (80:20)
train_files, val_files = train_test_split(xml_files, test_size=0.2, random_state=42)
print(f"Train: {len(train_files)}개, Val: {len(val_files)}개")

In [ ]:
# XML → YOLO 변환 및 파일 복사
success_count = 0
error_count = 0

for split, files in [('train', train_files), ('val', val_files)]:
    for xml_path in files:
        try:
            # 이미지 경로 찾기 (XML과 같은 이름, 다른 폴더)
            xml_name = os.path.splitext(os.path.basename(xml_path))[0]
            
            # images 폴더에서 이미지 찾기
            img_path = None
            for ext in ['.png', '.jpg', '.jpeg']:
                # data/images/ 폴더 확인
                test_path = f'data/images/{xml_name}{ext}'
                if os.path.exists(test_path):
                    img_path = test_path
                    break
                # XML과 같은 폴더 확인
                test_path = xml_path.replace('.xml', ext)
                if os.path.exists(test_path):
                    img_path = test_path
                    break

            if img_path and os.path.exists(img_path):
                # 이미지 복사
                img_name = os.path.basename(img_path)
                shutil.copy(img_path, f'dataset/images/{split}/{img_name}')

                # YOLO 라벨 변환 및 저장
                yolo_label = convert_xml_to_yolo(xml_path)
                label_name = xml_name + '.txt'
                with open(f'dataset/labels/{split}/{label_name}', 'w') as f:
                    f.write(yolo_label)

                success_count += 1
            else:
                print(f"이미지 없음: {xml_name}")
                error_count += 1

        except Exception as e:
            print(f"Error: {xml_path} - {e}")
            error_count += 1

print(f"\n✅ 데이터셋 준비 완료!")
print(f"성공: {success_count}개, 실패: {error_count}개")
print(f"Train 이미지: {len(os.listdir('dataset/images/train'))}개")
print(f"Val 이미지: {len(os.listdir('dataset/images/val'))}개")

## 5. dataset.yaml 생성

In [ ]:
yaml_content = """path: /content/dataset
train: images/train
val: images/val

nc: 1
names: ['plate']
"""

with open('dataset/dataset.yaml', 'w') as f:
    f.write(yaml_content)

print("dataset.yaml created!")
print(yaml_content)

## 6. YOLO 학습

In [ ]:
from ultralytics import YOLO

# YOLOv8n 모델 로드 (가장 가벼운 버전)
model = YOLO('yolov8n.pt')

# 학습 시작 (약 15-30분 소요)
results = model.train(
    data='dataset/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='plate_detector',
    patience=10
)

print("Training complete!")

## 7. 결과 확인

In [ ]:
from IPython.display import Image, display

# 학습 곡선
print("Training curves:")
display(Image(filename='runs/detect/plate_detector/results.png', width=800))

In [ ]:
# 예측 샘플
print("Prediction samples:")
display(Image(filename='runs/detect/plate_detector/val_batch0_pred.jpg', width=800))

## 8. 모델 테스트

In [ ]:
# 최고 성능 모델 로드
best_model = YOLO('runs/detect/plate_detector/weights/best.pt')

# 검증 이미지로 테스트
val_images = os.listdir('dataset/images/val')[:5]

for img_name in val_images:
    img_path = f'dataset/images/val/{img_name}'
    results = best_model(img_path)
    print(f"{img_name}: {len(results[0].boxes)} plates detected")

In [ ]:
# 결과 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, img_name in zip(axes, val_images[:3]):
    img_path = f'dataset/images/val/{img_name}'
    results = best_model(img_path)

    # 결과 이미지
    result_img = results[0].plot()
    ax.imshow(result_img)
    ax.set_title(f"{len(results[0].boxes)} plates")
    ax.axis('off')

plt.tight_layout()
plt.show()

## 9. 모델 저장

In [ ]:
# Google Drive에 저장
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/AI_Practice/03-LicensePlate/models'
os.makedirs(save_dir, exist_ok=True)

shutil.copy('runs/detect/plate_detector/weights/best.pt', f'{save_dir}/plate_detector.pt')
print(f"Model saved to: {save_dir}/plate_detector.pt")

In [ ]:
# 또는 직접 다운로드
from google.colab import files
files.download('runs/detect/plate_detector/weights/best.pt')

---

## 완료!

Step 1 (번호판 검출) 모델 학습이 완료되었습니다.

다음 단계:
- **Step 2**: 검출된 번호판에서 문자 분할
- **Step 3**: 분할된 문자 인식